# Domain 2: Tool Design & MCP Integration
## TS 2.3: Distribute Tools Appropriately and Configure Tool Choice
**Certification Focus**: CCAR-F Domain 2 (18% Exam Weight) | **Scenario Alignment**: Scenario 4 (Developer Productivity with Claude)

---

### 🎯 Key Architectural Principles
1. **Principle of Least Privilege**: Limit each agent's tool suite to 4–5 specialized tools. Giving agents 20+ tools degrades tool selection accuracy and increases token cost.
2. **Tool Choice Configuration**:
   - `"auto"`: Model freely decides whether to invoke tools or respond directly with text.
   - `"any"`: Model **must** call a tool, but selects which one.
   - `{"type": "tool", "name": "..."}`: Model is strictly forced to call a specific named tool.
3. **Role-Based Tool Distribution**: Coordinator agents receive delegation tools (`Task`), whereas worker agents receive domain-specific tools (`read_file`, `calculator`).

In [ ]:
# Step 1: Environment & Setup
import fnmatch
import os
import re
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 2.3 Tool Distribution & Choice.")

### 🎛️ Step 2: Tool Choice Resolver & Built-in Tools Engine
We implement the tool choice configuration parser and built-in search engine.

In [ ]:
# Step 2: Tool Choice Resolver Logic
def resolve_tool_choice_mode(mode_str: str, specific_tool_name: Optional[str] = None) -> Any:
    """Resolves user configuration into the official Anthropic tool_choice parameter."""
    normalized = mode_str.lower().strip()
    if normalized == "auto":
        return "auto"
    elif normalized == "any":
        return {"type": "any"}
    elif normalized == "forced":
        if not specific_tool_name:
            raise ValueError("Forced mode requires specific_tool_name")
        return {"type": "tool", "name": specific_tool_name}
    raise ValueError(f"Unknown tool_choice mode: {mode_str}")


class BuiltinToolsEngine:
    """Simulates developer productivity tools (Glob, Grep) over a directory."""
    def __init__(self, workspace_root: Path) -> None:
        self.root = workspace_root

    def glob_files(self, pattern: str) -> List[str]:
        matches = []
        ignore_dirs = {".git", ".venv", "venv", "__pycache__", ".pytest_cache", ".mypy_cache", ".ruff_cache"}
        for dirpath, dirnames, filenames in os.walk(str(self.root)):
            dirnames[:] = [d for d in dirnames if d not in ignore_dirs and not d.startswith(".venv")]
            rel_dir = os.path.relpath(dirpath, str(self.root)).replace("\\", "/")
            for fname in filenames:
                rel = f"{rel_dir}/{fname}" if rel_dir != "." else fname
                if fnmatch.fnmatch(rel, pattern) or fnmatch.fnmatch(fname, pattern):
                    matches.append(rel)
        return sorted(matches)

    def grep_content(self, regex_pattern: str, file_pattern: str = "*") -> List[Dict[str, Any]]:
        results = []
        compiled = re.compile(regex_pattern)
        for rel_path in self.glob_files(file_pattern):
            full_path = self.root / rel_path
            try:
                lines = full_path.read_text(encoding="utf-8").splitlines()
                for line_no, line in enumerate(lines, start=1):
                    if compiled.search(line):
                        results.append({"file": rel_path, "line": line_no, "content": line.strip()})
            except Exception:
                continue
        return results

print(" ToolChoice resolver and BuiltinToolsEngine ready.")

### 🧪 Step 3: Interactive Simulation
We test tool_choice resolution modes and simulate file exploration with glob and grep.

In [ ]:
# Step 3: Simulation
# 1. Test tool_choice modes
mode_auto = resolve_tool_choice_mode("auto")
mode_any = resolve_tool_choice_mode("any")
mode_forced = resolve_tool_choice_mode("forced", "security_audit")

print("tool_choice 'auto':", mode_auto)
print("tool_choice 'any':", mode_any)
print("tool_choice 'forced':", mode_forced)

# 2. Test BuiltinToolsEngine on repository files
engine = BuiltinToolsEngine(repo_root)
py_files = engine.glob_files("*.md")
print(f"\nGlobbed Markdown files in repo root: {len(py_files)} files found.")

# Grep for CCAR-F mentions in markdown files
grep_res = engine.grep_content("CCAR-F", "*.md")
print(f"Grep matched 'CCAR-F' in {len(grep_res)} line(s).")

### 🔬 Step 4: Verification Assertions
We assert that:
1. `resolve_tool_choice_mode` outputs the exact Anthropic schema.
2. Glob and grep correctly locate files and strings.

In [ ]:
# Step 4: Verification Checks
assert mode_auto == "auto"
assert mode_any == {"type": "any"}
assert mode_forced == {"type": "tool", "name": "security_audit"}

assert len(py_files) > 0
assert len(grep_res) > 0

print(" All verification assertions passed! Tool distribution verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Assign focused tool sets (4–5 tools per subagent) conforming to least privilege.
- **Rule 2**: Use `tool_choice: {"type": "tool", "name": "..."}` to force mandatory verification steps.
- **Rule 3**: Combine Glob (file discovery) and Grep (symbol search) for token-efficient codebase exploration.